# LeetCode 3116 — Kth Smallest Amount With Single Denomination Combination bằng Python

- Link: https://leetcode.com/problems/kth-smallest-amount-with-single-denomination-combination/
- Độ khó: **Hard**
- Chủ đề: **Tìm nhị phân trên đáp án** + đếm số `≤ x` chia hết cho **ít nhất một** đồng xu bằng **bao hàm – loại trừ** trên các tập con (dùng **BCNN**); tối ưu: bỏ đồng xu là bội của đồng khác, gộp tập con cùng BCNN

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findKthSmallest(self, coins: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Các số tạo được là **hợp các tập bội** `{c, 2c, 3c, …}` của từng đồng xu (không trộn mệnh giá).
2. `count(x)` = số nguyên dương `≤ x` chia hết cho ít nhất một `c` — đơn điệu theo `x` ⇒ tìm nhị phân `x` nhỏ nhất với `count(x) ≥ k`.
3. Bao hàm – loại trừ: `count(x) = Σ_{S ≠ ∅} (−1)^{|S|+1} ⌊x / lcm(S)⌋` — tối đa `2^{15} − 1` tập con.
4. Tối ưu: bỏ các đồng xu là bội của đồng khác (không thêm số mới); gộp các tập con có cùng `lcm` (cộng dồn hệ số dấu).
5. Kiểm chứng bằng trộn heap các dãy bội (brute).

## 2. Đề bài & ví dụ

Cho mảng `coins` (mệnh giá khác nhau) và `k`. Mỗi mệnh giá có vô hạn đồng nhưng **không được trộn** mệnh giá: số tiền tạo được là bội dương của một mệnh giá nào đó. Trả về số tiền nhỏ thứ `k` (các số tiền khác nhau).

**Ví dụ 1:** `coins = [3,6,9]`, `k = 3` → `9`. **Ví dụ 2:** `coins = [5,2]`, `k = 7` → `12`.

**Ràng buộc:** `1 ≤ coins.length ≤ 15`; `1 ≤ coins[i] ≤ 25`; `1 ≤ k ≤ 2·10⁹`; các mệnh giá khác nhau.

In [1]:
from typing import List
from math import gcd
import heapq, itertools, random, time

examples = [([3, 6, 9], 3, 9), ([5, 2], 7, 12)]
for c, k, exp in examples:
    print(c, 'k =', k, '-> kỳ vọng', exp)

[3, 6, 9] k = 3 -> kỳ vọng 9
[5, 2] k = 7 -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Bao hàm – loại trừ với BCNN

`x` chia hết cho mọi phần tử của `S` ⇔ chia hết cho `lcm(S)`. Số nguyên trong `[1, x]` thuộc **hợp** các tập bội:

`|A_1 ∪ … ∪ A_t| = Σ_i |A_i| − Σ_{i<j} |A_i ∩ A_j| + …`, với `|∩_{i∈S} A_i| = ⌊x / lcm(S)⌋`.

In [2]:
def lcm(a, b):
    return a // gcd(a, b) * b

def count_ie(coins, x):
    t = len(coins); total = 0
    for mask in range(1, 1 << t):
        L = 1; bits = 0
        for i in range(t):
            if mask >> i & 1:
                L = lcm(L, coins[i]); bits += 1
        total += (x // L) if bits & 1 else -(x // L)
    return total

for c, _, _ in examples:
    print(c, [count_ie(c, x) for x in range(1, 16)], '| trực tiếp:', [sum(1 for y in range(1, x + 1) if any(y % q == 0 for q in c)) for x in range(1, 16)])

[3, 6, 9] [0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5] | trực tiếp: [0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5]
[5, 2] [0, 1, 1, 2, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 9] | trực tiếp: [0, 1, 1, 2, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 9]


### 3.2. Tìm nhị phân "giá trị nhỏ nhất thoả"

Số thứ `k` là `x` nhỏ nhất với `count(x) ≥ k`: vì `count` tăng đúng 1 tại mỗi số tạo được và đứng yên ở các số khác. Cận trên: `min(coins) · k` (riêng đồng nhỏ nhất đã cho `k` số `≤` giá trị đó).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`k` tới `2·10⁹` ⇒ không liệt kê được. Nhưng "có bao nhiêu số tiền `≤ x`" tính nhanh bằng bao hàm – loại trừ (vì chỉ có tối đa 15 mệnh giá). Ví dụ 2: `count(12) = 6 + 2 − 1 = 7` (bội của 2: 6 số, bội của 5: 2 số, bội của 10: 1 số) và `count(11) = 5 + 2 − 1 = 6` ⇒ số thứ 7 là `12`.

## 5. Cách 1 — Brute Force: trộn các dãy bội bằng heap

In [3]:
def brute(coins, k):
    heap = [(c, c) for c in coins]; heapq.heapify(heap); last = 0; cnt = 0
    while True:
        v, c = heapq.heappop(heap)
        heapq.heappush(heap, (v + c, c))
        if v != last:
            cnt += 1; last = v
            if cnt == k:
                return v

for c, k, exp in examples:
    print('heap:', brute(c, k), '| kỳ vọng', exp)

heap: 9 | kỳ vọng 9
heap: 12 | kỳ vọng 12


**Độ phức tạp:** `O(k · t log t)` — chỉ cho `k` nhỏ (vài chục nghìn).

## 6. Ý tưởng thuật toán / bước đột phá

```
lọc coins: bỏ c nếu có c' < c với c % c' == 0
terms = [(lcm(S), dấu(S))] cho mọi tập con khác rỗng (bỏ lcm > cận trên)
lo, hi = 1, min(coins) * k
while lo < hi: mid; nếu Σ dấu·(mid // lcm) >= k: hi = mid, ngược lại lo = mid + 1
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tìm nhị phân + bao hàm – loại trừ thuần

In [4]:
def binary_ie(coins, k):
    lo, hi = 1, min(coins) * k
    while lo < hi:
        mid = (lo + hi) // 2
        if count_ie(coins, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for c, k, exp in examples:
    print('nhị phân + IE:', binary_ie(c, k), '| kỳ vọng', exp)

nhị phân + IE: 9 | kỳ vọng 9
nhị phân + IE: 12 | kỳ vọng 12


Mỗi bước tìm nhị phân duyệt lại `2^t` tập con và tính lại BCNN — chậm (`36 · 32768 · 15 ≈ 1.8·10⁷`). Cách 3 tính trước.

### 7.2. Cách 3 — Lọc đồng xu, tính trước và **gộp** các BCNN (lời giải chính)

In [5]:
def precomputed_terms(coins, cap):
    cs = sorted(coins)
    cs = [c for i, c in enumerate(cs) if all(c % d for d in cs[:i])]   # bỏ bội của đồng nhỏ hơn
    coef = {}                         # coef[L] = Σ_{S ≠ ∅, lcm(S) = L} (−1)^{|S|+1}
    for c in cs:
        new = dict(coef)
        new[c] = new.get(c, 0) + 1    # tập {c}
        for L, w in coef.items():     # S ∪ {c}: đổi dấu
            L2 = lcm(L, c)
            if L2 <= cap:
                new[L2] = new.get(L2, 0) - w
        coef = new
    return [(L, w) for L, w in coef.items() if w != 0]

def merged(coins, k):
    hi = min(coins) * k; terms = precomputed_terms(coins, hi); lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(w * (mid // L) for L, w in terms) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print('các số hạng (lcm, hệ số) cho [3, 6, 9]:', precomputed_terms([3, 6, 9], 100), '| cho [5, 2]:', precomputed_terms([5, 2], 100))
for c, k, exp in examples:
    print('gộp BCNN:', merged(c, k), '| kỳ vọng', exp)

các số hạng (lcm, hệ số) cho [3, 6, 9]: [(3, 1)] | cho [5, 2]: [(2, 1), (5, 1), (10, -1)]
gộp BCNN: 9 | kỳ vọng 9
gộp BCNN: 12 | kỳ vọng 12


Giải thích hệ số: `coef[L] = Σ (−1)^{|S|+1}` trên các tập con **khác rỗng** `S` có `lcm(S) = L`. Khi xét thêm đồng `c`: các tập mới là `{c}` (hệ số `+1` vào ô `c`) và `S ∪ {c}` với mọi tập cũ `S` (đổi dấu, vào ô `lcm(L, c)`). Gộp theo `L` làm số số hạng giảm mạnh, và bỏ `L > cận trên` vì `⌊x / L⌋ = 0`.

**Cạm bẫy khi cài:** một cách "gọn" là khởi đầu bằng tập rỗng `{1: −1}` rồi cuối cùng xoá ô `1`. Nhưng nếu có đồng mệnh giá `1` thì tập `{1}` **cũng** có BCNN bằng 1 — hai hệ số `−1` và `+1` triệt tiêu nhau trong cùng ô, và xoá ô `1` làm mất luôn số hạng thật! Ô dưới minh hoạ.

In [6]:
def buggy_terms(coins, cap):
    cs = sorted(coins); cs = [c for i, c in enumerate(cs) if all(c % d for d in cs[:i])]
    coef = {1: -1}
    for c in cs:
        new = dict(coef)
        for L, w in coef.items():
            L2 = lcm(L, c)
            if L2 <= cap:
                new[L2] = new.get(L2, 0) - w
        coef = new
    coef.pop(1, None)
    return [(L, w) for L, w in coef.items() if w != 0]

print('coins = [1, 3]: số hạng đúng =', precomputed_terms([1, 3], 100), '| bản lỗi =', buggy_terms([1, 3], 100))
print('coins = [2, 3]: số hạng đúng =', precomputed_terms([2, 3], 100), '| bản lỗi =', buggy_terms([2, 3], 100))

coins = [1, 3]: số hạng đúng = [(1, 1)] | bản lỗi = []
coins = [2, 3]: số hạng đúng = [(2, 1), (3, 1), (6, -1)] | bản lỗi = [(2, 1), (3, 1), (6, -1)]


### 7.3. Cách 4 — Đếm theo chu kỳ BCNN toàn bộ

Nếu `P = lcm(coins)` nhỏ, các số tạo được lặp lại theo chu kỳ `P`: liệt kê `r` số tạo được trong `[1, P]` (sắp tăng), thì số thứ `k` là `((k − 1) // r) · P + list[(k − 1) % r]`. Chỉ dùng được khi `P` vừa phải (ví dụ `≤ 10^6`).

In [7]:
def by_period(coins, k, limit=10 ** 6):
    P = 1
    for c in coins:
        P = lcm(P, c)
    if P > limit:
        return None
    period = [y for y in range(1, P + 1) if any(y % c == 0 for c in coins)]
    r = len(period)
    return (k - 1) // r * P + period[(k - 1) % r]

for c, k, exp in examples:
    print('theo chu kỳ:', by_period(c, k), '| kỳ vọng', exp)

theo chu kỳ: 9 | kỳ vọng 9
theo chu kỳ: 12 | kỳ vọng 12


## 8. TRACE từng bước

Trace tìm nhị phân của cách 3 trên ví dụ 2.

In [8]:
coins, k = examples[1][:2]
hi = min(coins) * k; terms = precomputed_terms(coins, hi); lo = 1
print('số hạng:', terms, '| khoảng tìm [1,', hi, ']')
while lo < hi:
    mid = (lo + hi) // 2; c = sum(w * (mid // L) for L, w in terms)
    print(f'  mid = {mid:2d}: count = {c} -> {"hi = mid" if c >= k else "lo = mid + 1"}')
    if c >= k: hi = mid
    else: lo = mid + 1
print('đáp án:', lo)

số hạng: [(2, 1), (5, 1), (10, -1)] | khoảng tìm [1, 14 ]
  mid =  7: count = 4 -> lo = mid + 1
  mid = 11: count = 6 -> lo = mid + 1
  mid = 13: count = 7 -> hi = mid
  mid = 12: count = 7 -> hi = mid
đáp án: 12


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def findKthSmallest(self, coins: List[int], k: int) -> int:
        cs = sorted(coins)
        cs = [c for i, c in enumerate(cs) if all(c % d for d in cs[:i])]
        hi = cs[0] * k
        coef = {}
        for c in cs:
            new = dict(coef)
            new[c] = new.get(c, 0) + 1
            for L, w in coef.items():
                L2 = L // gcd(L, c) * c
                if L2 <= hi:
                    new[L2] = new.get(L2, 0) - w
            coef = new
        terms = [(L, w) for L, w in coef.items() if w]
        lo = 1
        while lo < hi:
            mid = (lo + hi) // 2
            cnt = 0
            for L, w in terms:
                cnt += w * (mid // L)
            if cnt >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

for c, k, exp in examples:
    got = Solution().findKthSmallest(c, k)
    print(got, 'OK' if got == exp else 'SAI')

9 OK
12 OK


## 10. Vì sao thuật toán đúng

1. **Đếm:** `count(x)` đúng theo bao hàm – loại trừ (mục 3.1); bỏ đồng `c` là bội của `d` không đổi hợp vì `A_c ⊆ A_d`; bỏ số hạng `L > hi` không đổi giá trị với mọi `x ≤ hi`.
2. **Tìm nhị phân:** `count` không giảm, `count(hi) ≥ k` (với `hi = min · k`), và số thứ `k` là `x` nhỏ nhất có `count(x) ≥ k` (mục 3.2).
3. **Gộp hệ số:** mỗi tập con `S` đóng góp `(−1)^{|S|+1}` vào ô `lcm(S)`; quy trình "thêm đồng `c` vào mọi tập hiện có và đổi dấu" liệt kê đúng mọi tập con (mỗi tập một lần).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Trộn heap | `O(k t log t)` | `O(t)` |
| 2. Nhị phân + IE thuần | `O(log(hi) · 2^t · t)` | `O(1)` |
| 3. Lọc + gộp BCNN | `O(2^t + log(hi) · |terms|)` | `O(|terms|)` |
| 4. Theo chu kỳ | `O(P · t)` | `O(P)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'binary_ie': binary_ie, 'merged': merged, 'submit': lambda c, k: Solution().findKthSmallest(c, k)}
edge = [([1], 1, 1), ([1], 10, 10), ([7], 5, 35), ([2, 3], 4, 6), ([4, 6], 3, 8), ([2, 4, 8], 5, 10), ([25, 24], 3, 48)]
for c, k, exp in examples + edge:
    assert brute(c, k) == exp, (c, k, brute(c, k))
    for name, f in impls.items():
        assert f(c, k) == exp, (name, c, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[4, 6]`, `k = 3`: các số `4, 6, 8, 12, …` ⇒ `8`. `[2, 4, 8]`: chỉ cần đồng 2 ⇒ `2k = 10`. `[25, 24]`: `24, 25, 48, 50, …` ⇒ thứ 3 là `48`.

### 12.2. Đối chiếu ngẫu nhiên với heap

In [11]:
rng = random.Random(3116)
cnt = 0
for _ in range(400):
    t = rng.randint(1, 6); c = rng.sample(range(1, 26), t); k = rng.randint(1, 3000)
    exp = brute(c, k)
    for name, f in impls.items():
        assert f(c, k) == exp, (name, c, k)
    p = by_period(c, k)
    assert p is None or p == exp
    cnt += 1
print('Khớp heap trên', cnt, 'bộ ngẫu nhiên (và cách chu kỳ khi áp dụng được).')

Khớp heap trên 400 bộ ngẫu nhiên (và cách chu kỳ khi áp dụng được).


### 12.3. Ca lớn: 15 mệnh giá, `k = 2·10⁹`

In [12]:
big = [(list(range(11, 26)), 2 * 10 ** 9), (rng.sample(range(1, 26), 15), 2 * 10 ** 9),
       ([2, 3, 5, 7, 11, 13, 17, 19, 23, 25, 22, 21, 9, 8, 1], 2 * 10 ** 9)]
for c, k in big:
    t = time.time(); x = Solution().findKthSmallest(c, k); t1 = time.time() - t
    t = time.time(); y = merged(c, k); t2 = time.time() - t
    assert x == y
    print(f'coins = {c} -> {x} | nộp {t1:.2f}s | gộp {t2:.2f}s | số hạng = {len(precomputed_terms(c, min(c) * k))}')
assert Solution().findKthSmallest([2, 3, 5, 7, 11, 13, 17, 19, 23, 25, 22, 21, 9, 8, 1], 2 * 10 ** 9) == 2 * 10 ** 9
print('Có đồng 1 ⇒ mọi số dương tạo được ⇒ đáp án = k: đúng.')

coins = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25] -> 3980739261 | nộp 0.00s | gộp 0.00s | số hạng = 1151
coins = [18, 8, 14, 13, 12, 21, 3, 16, 7, 17, 10, 11, 5, 23, 19] -> 2802216768 | nộp 0.00s | gộp 0.00s | số hạng = 511
coins = [2, 3, 5, 7, 11, 13, 17, 19, 23, 25, 22, 21, 9, 8, 1] -> 2000000000 | nộp 0.00s | gộp 0.00s | số hạng = 1
Có đồng 1 ⇒ mọi số dương tạo được ⇒ đáp án = k: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `Σ ⌊x / c⌋` (không loại phần giao)

In [13]:
def wrong_no_ie(coins, k):
    lo, hi = 1, min(coins) * k
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // c for c in coins) >= k: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_no_ie(*examples[1][:2]), '| đúng:', Solution().findKthSmallest(*examples[1][:2]))

sai : 10 | đúng: 12


**Vì sao sai:** số `10` vừa là bội của 2 vừa của 5 — bị đếm hai lần, đáp án bị kéo xuống.

### Lỗi 2 — Dùng **tích** thay cho BCNN ở phần giao

In [14]:
def wrong_product(coins, k):
    t = len(coins)
    def cnt(x):
        s = 0
        for mask in range(1, 1 << t):
            P = 1; b = 0
            for i in range(t):
                if mask >> i & 1:
                    P *= coins[i]; b += 1
            s += (x // P) if b & 1 else -(x // P)
        return s
    lo, hi = 1, min(coins) * k
    while lo < hi:
        mid = (lo + hi) // 2
        if cnt(mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_product(*examples[0][:2]), '| đúng:', Solution().findKthSmallest(*examples[0][:2]))

sai : 6 | đúng: 9


**Vì sao sai:** bội chung của 3 và 6 là bội của `lcm = 6`, không phải của `18`. Tích chỉ đúng khi các mệnh giá đôi một nguyên tố cùng nhau.

### Lỗi 3 — Tìm nhị phân với điều kiện ngặt `count(mid) > k`

In [15]:
def wrong_strict(coins, k):
    lo, hi = 1, min(coins) * k + max(coins)
    while lo < hi:
        mid = (lo + hi) // 2
        if count_ie(coins, mid) > k: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_strict(*examples[1][:2]), '| đúng:', Solution().findKthSmallest(*examples[1][:2]))

sai : 14 | đúng: 12


**Vì sao sai:** trả về số tạo được thứ `k + 1`. Cần `x` nhỏ nhất với `count(x) ≥ k`.

### Lỗi 4 — Cận trên `hi = k` (quên nhân với mệnh giá)

In [16]:
def wrong_hi(coins, k):
    lo, hi = 1, k                                                      # SAI
    while lo < hi:
        mid = (lo + hi) // 2
        if count_ie(coins, mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_hi(*examples[1][:2]), '| đúng:', Solution().findKthSmallest(*examples[1][:2]))

sai : 7 | đúng: 12


**Vì sao sai:** số thứ `k` có thể lớn hơn `k` nhiều (tới `min(coins) · k`); khoảng tìm thiếu đáp án nên vòng lặp dừng ở `k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số nhỏ thứ `k` trong hợp các cấp số cộng / tập bội" ⇒ tìm nhị phân + đếm bao hàm – loại trừ (LC 878, 1201).
- `t ≤ 15` ⇒ `2^t` tập con chấp nhận được; gộp theo BCNN để nhanh hơn.
- Luôn dùng BCNN cho phần giao các tập bội.

## 15. Biến thể của bài

1. **LC 878 Nth Magical Number:** hai mệnh giá.
2. **LC 1201 Ugly Number III:** ba mệnh giá.
3. Được **trộn** mệnh giá ⇒ bài toán đồng xu Frobenius — khác hẳn (DP / Dijkstra theo modulo).
4. Số nhỏ thứ `k` **không** tạo được (chia hết cho không đồng nào) ⇒ `x − count(x)`.

In [17]:
# Biến thể 4: số nguyên dương nhỏ thứ k KHÔNG chia hết cho đồng nào (yêu cầu không có đồng 1)
def kth_not_made(coins, k):
    lo, hi = 1, k * max(2, max(coins)) * 2
    while lo < hi:
        mid = (lo + hi) // 2
        if mid - count_ie(coins, mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

for _ in range(200):
    c = rng.sample(range(2, 12), rng.randint(1, 4)); k = rng.randint(1, 200)
    seq = [y for y in range(1, 100000) if all(y % q for q in c)]
    assert kth_not_made(c, k) == seq[k - 1]
assert Solution().findKthSmallest([2, 3], 10 ** 9) == 1500000000 and Solution().findKthSmallest([2, 3, 5], 3) == 4   # LC 878 / LC 1201 dạng
print('Biến thể "không tạo được" khớp liệt kê; các dạng LC 878/1201 đúng.')

Biến thể "không tạo được" khớp liệt kê; các dạng LC 878/1201 đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Trộn heap | `O(k t log t)` | Oracle |
| 2 | Nhị phân + IE thuần | `O(log · 2^t · t)` | Đơn giản |
| 3 | Lọc + gộp BCNN | `O(2^t + log · terms)` | **Nộp bài** |
| 4 | Chu kỳ BCNN | `O(P t)` | Khi `P` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `coins = [4]`, `k = 6`.

**Bài 2 (tính tay):** `coins = [3, 5]`, `k = 6`.

**Bài 3 (điền TODO):** trả về `k` số tiền nhỏ nhất đầu tiên (danh sách), `k ≤ 10^5`.

**Bài 4 (tự viết lại):** đếm số tiền tạo được trong đoạn `[a, b]`.

**Bài 5 (mini challenge):** cho thêm điều kiện số đồng dùng **tối đa `q`** đồng (tức số tiền `= c · j` với `j ≤ q`) — tìm số nhỏ thứ `k`.

In [18]:
def first_k_todo(coins, k):
    # TODO
    raise NotImplementedError

def count_range_todo(coins, a, b):
    # TODO
    raise NotImplementedError

def limited_count_todo(coins, k, q):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ bội của 4. (2) `4k`. (3) `24`.

**Bài 2.** (1) `3, 5, 6, 9, 10, 12, 15, …`. (2) Thứ 6 là `12`. (3) Kiểm `count(12) = 4 + 2 − 0 = 6`.

**Bài 3.** (1) Trộn heap như brute. (2) Bỏ trùng. (3) Dừng khi đủ `k`.

**Bài 4.** (1) `count(b) − count(a − 1)`. (2) Dùng số hạng đã gộp. (3) Kiểm liệt kê.

**Bài 5.** (1) Số tiền `y` tạo được ⇔ tồn tại `c | y` với `y / c ≤ q` ⇔ `y ≤ q·c`. (2) `count(x) = |∪ {c·j : j ≤ min(q, x/c)}|` — bao hàm – loại trừ với `⌊min(x, q·min_S)/lcm(S)⌋`? cần cẩn thận: phần giao `S` gồm các `y` chia hết `lcm(S)` và `y ≤ q·c` với mọi `c ∈ S` ⇒ `y ≤ q · min(S)`. (3) Tìm nhị phân như cũ, trả `-1` nếu không đủ `k` số.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3, 4
assert Solution().findKthSmallest([4], 6) == 24 and Solution().findKthSmallest([3, 5], 6) == 12

def first_k(coins, k):
    heap = [(c, c) for c in coins]; heapq.heapify(heap); out = []
    while len(out) < k:
        v, c = heapq.heappop(heap); heapq.heappush(heap, (v + c, c))
        if not out or out[-1] != v:
            out.append(v)
    return out

def count_range(coins, a, b):
    return count_ie(coins, b) - count_ie(coins, a - 1)

for _ in range(200):
    c = rng.sample(range(1, 26), rng.randint(1, 5)); k = rng.randint(1, 300)
    lst = first_k(c, k)
    assert lst[-1] == Solution().findKthSmallest(c, k) and lst == sorted(set(lst))
    a = rng.randint(1, 500); b = rng.randint(a, a + 500)
    assert count_range(c, a, b) == sum(1 for y in range(a, b + 1) if any(y % q == 0 for q in c))
print('Bài 1: 24 | Bài 2: 12 | Bài 3, 4 khớp liệt kê.')

Bài 1: 24 | Bài 2: 12 | Bài 3, 4 khớp liệt kê.


In [20]:
# Bài 5: tối đa q đồng
def limited(coins, k, q):
    t = len(coins)
    def cnt(x):
        s = 0
        for mask in range(1, 1 << t):
            L = 1; b = 0; mn = 10 ** 18
            for i in range(t):
                if mask >> i & 1:
                    L = lcm(L, coins[i]); b += 1; mn = min(mn, coins[i])
            v = min(x, q * mn) // L
            s += v if b & 1 else -v
        return s
    hi = q * max(coins)
    if cnt(hi) < k:
        return -1
    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if cnt(mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

for _ in range(200):
    c = rng.sample(range(1, 15), rng.randint(1, 4)); q = rng.randint(1, 20); k = rng.randint(1, 60)
    vals = sorted({cc * j for cc in c for j in range(1, q + 1)})
    assert limited(c, k, q) == (vals[k - 1] if k <= len(vals) else -1)
print('Bài 5 (tối đa q đồng) khớp liệt kê.')

Bài 5 (tối đa q đồng) khớp liệt kê.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cs = các đồng không là bội của đồng nhỏ hơn
coef = {1: -1}; với mỗi c: thêm lcm(L, c) với hệ số −coef[L] (bỏ L > hi)
bỏ coef[1]; tìm nhị phân x nhỏ nhất với Σ w·(x // L) >= k, hi = min(coins)·k
```

**Checklist:**

- [ ] Phần giao dùng **BCNN**?
- [ ] Bao hàm – loại trừ đúng dấu (lẻ +, chẵn −)?
- [ ] Điều kiện tìm nhị phân `≥ k`?
- [ ] Cận trên `min(coins) · k`?